# Embeddings

In [2]:
pip install fastembed numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from fastembed import TextEmbedding
import numpy as np

# Q1. Embedding the query

In [4]:
# loada the embedding model
model = TextEmbedding(model_name="jinaai/jina-embeddings-v2-small-en")

In [5]:
# embed the query
query = 'I just discovered the course. Can I join now?'
embeddding = list(model.embed([query]))[0] # return a list of embeddings

In [6]:
# check the shape of the embedding
print(f"Shape of the embedding: {embeddding.shape}")

Shape of the embedding: (512,)


In [7]:
# find the minimum value in the array
min_value = np.min(embeddding)
print(f"Minimum value in the embedding: {min_value}")

Minimum value in the embedding: -0.11726373885183883


# Q2. Cosine similarity with another vector

**Cosine similarity**

The vectors that our embedding model returns are already normalized: their length is 1.0.

You can check that by using the norm function:

```
import numpy as np
np.linalg.norm(q)
```

Which means that we can simply compute the dot product between two vectors to learn the cosine similarity between them.

For example, if you compute the cosine of the query vector with itself, the result will be 1.0:

`q.dot(q)`

---
Cosine Similarity = (A ⋅ B) / (||A|| ⋅ ||B||)

Where:
- A and B are the two vectors. 
- A ⋅ B is the dot product of the vectors. 
- ||A|| and ||B|| are the magnitudes (lengths) of the vectors. 

| Step | What it does                   |
| ---- | ------------------------------ |
| 1    | Install required libraries     |
| 2    | Import `fastembed` and `numpy` |
| 3    | Embed query & doc              |
| 4    | Define cosine similarity       |
| 5    | Calculate and print result     |


In [8]:
# embed the two texts
query = "I just discovered the course. Can I join now?"
doc = 'Can I still join the course after the start date?'

query_vector = list(model.embed([query]))[0]
doc_vector = list(model.embed([doc]))[0]

In [9]:
# define cosine similarity function
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

In [10]:
# Compute cosine similarity
similarity = cosine_similarity(query_vector, doc_vector)
print(f"Cosine similarity between query and doc: {similarity}")

Cosine similarity between query and doc: 0.9008528895674548


# Q3. Ranking by cosine

| Step | Action                                       |
| ---- | -------------------------------------------- |
| 1-2  | Import and install tools                     |
| 3    | Define data                                  |
| 4    | Load the FastEmbed model                     |
| 5-6  | Embed all docs and query                     |
| 7    | Use matrix math to compute cosine similarity |
| 8    | Use `argmax` to find most similar one        |


In [11]:
documents = [{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.',
  'section': 'General course-related questions',
  'question': 'Course - Can I follow the course after it finishes?',
  'course': 'data-engineering-zoomcamp'},
 {'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
  'section': 'General course-related questions',
  'question': 'Course - When will the course start?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'You can start by installing and setting up all the dependencies and requirements:\nGoogle cloud account\nGoogle Cloud SDK\nPython 3 (installed with Anaconda)\nTerraform\nGit\nLook over the prerequisites and syllabus to see if you are comfortable with these subjects.',
  'section': 'General course-related questions',
  'question': 'Course - What can I do before the course starts?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Star the repo! Share it with friends if you find it useful ❣️\nCreate a PR if you see you can improve the text or the structure of the repository.',
  'section': 'General course-related questions',
  'question': 'How can we contribute to the course?',
  'course': 'data-engineering-zoomcamp'}]

In [12]:
query = "Can I still join the course after the start date?"

In [13]:
# embed all texts fields into a 2D array
doc_texts = [doc['text'] for doc in documents]
doc_embeddings = np.array(list(model.embed(doc_texts))) # shape: (n_docs, embedding_dim) = (5,512)

In [14]:
# embed the query
query_embedding = np.array(list(model.embed([query]))[0]) # shape: (embedding_dim,) = (512,)

In [15]:
# nomalize the embeddings
docs_norm = np.linalg.norm(doc_embeddings, axis=1)
query_norm = np.linalg.norm(query_embedding)

# compute cosine similarities
cosine_similarities = np.dot(doc_embeddings, query_embedding) / (docs_norm * query_norm)

In [16]:
# highest similarity

best_match_index = np.argmax(cosine_similarities)
print(f"Best match index: {best_match_index}")

Best match index: 1


# Q4. Ranking by cosine, version two

In [25]:
# create full text for each document
full_texts = [doc['question'] + ' ' + doc['text'] for doc in documents]

In [26]:
doc_embeddings = np.array(list(model.embed(full_texts))) # shape: (n_docs, embedding_dim) = (5,512)

In [27]:
query_embedding = np.array(list(model.embed([query]))[0]) # shape: (embedding_dim,) = (512,)

In [28]:
# nomalize the embeddings
docs_norm = np.linalg.norm(doc_embeddings, axis=1)
query_norm = np.linalg.norm(query_embedding)

# compute cosine similarities
cosine_similarities = np.dot(doc_embeddings, query_embedding) / (docs_norm * query_norm)

In [29]:
# highest similarity

best_match_index = np.argmax(cosine_similarities)
print(f"Best match index: {best_match_index}")

Best match index: 0


Yes, it can be different from Q3 because:

In Q3, I only embedded the `"text"` field.

In Q4, I embedded `"question + text"` — this gives more semantic context, possibly making the match stronger or shifting it to a more relevant document.

The extra context helps the embedding model make more informed similarity comparisons.

# Q5. Selecting the embedding model

In [30]:
model = TextEmbedding(model_name="BAAI/bge-small-en")

In [33]:
# generate an embedding for any text
text = "Sample text for embedding"
embedding = list(model.embed([text]))[0]  
print(f"Embedding size: {len(embedding)}")

Embedding size: 384


# Q6. Indexing with qdrant

In [ ]:
pip install qdrant-client

In [35]:
import requests 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()


documents = []

for course in documents_raw:
    course_name = course['course']
    if course_name != 'machine-learning-zoomcamp':
        continue

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [40]:
# embed the data (using the BAAI model in Q5)
from fastembed import TextEmbedding
from qdrant_client import QdrantClient, models
from qdrant_client.models import PointStruct, VectorParams, Distance, CollectionStatus

model = TextEmbedding(model_name="BAAI/bge-small-en")  # 384 dimensions


In [ ]:
# prepare the data for Qdrant
collection_name = 'machine-learning-zoomcamp'
client = QdrantClient(url='http://localhost:6333')

In [43]:
collection_name = "ml_zoomcamp_faq"
client = QdrantClient(":memory:")  # Use in-memory Qdrant for quick experiments

# Create collection
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

# Prepare points for upload
points = []
id = 0

for doc in (documents):
    full_text = doc["question"] + " " + doc["text"]
    embedding = list(model.embed([full_text]))[0]

    point = models.PointStruct(
        id=id,
        vector=embedding,
        payload={
            "question": doc["question"],
            "text": doc["text"],
            "course": doc["course"]
        }
    )
    points.append(point)
    id += 1 

# Upload to Qdrant
client.upsert(collection_name=collection_name, points=points)


C:\Users\cong.vo\AppData\Local\Temp\ipykernel_24316\3010779574.py:5: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [38]:
query = "I just discovered the course. Can I join now?"


In [45]:
query_vector = list(model.embed([query]))[0]

search_result = client.search(
    collection_name=collection_name,
    query_vector=query_vector,
    limit=1
)

print("Top result score:", search_result[0].score)
print("Top result question:", search_result[0].payload['question'])


Top result score: 0.8703173398971558
Top result question: The course has already started. Can I still join it?


C:\Users\cong.vo\AppData\Local\Temp\ipykernel_24316\1450661731.py:3: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_result = client.search(
